In [ ]:
# Core FL and ML libraries
%pip install flwr[simulation] torch torchvision transformers datasets peft
# For efficiency during baseline training
%pip install bitsandbytes accelerate

In [ ]:
#base model 
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Note: For the baseline, we use standard LoRA (PEFT) to make training feasible
config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

In [ ]:
import torch
import flwr as fl

# Flower client for local training/eval
class HealthClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader, device="cpu"):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

    def get_parameters(self):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = {
            k: torch.tensor(v).to(self.device)
            for k, v in zip(self.model.state_dict().keys(), parameters)
        }
        self.model.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.Adam(self.model.parameters(), lr=float(config.get("lr", 1e-4)))
        loss_fn = torch.nn.CrossEntropyLoss()

        epochs = int(config.get("epochs", 1))
        for _ in range(epochs):
            for batch in self.train_loader:
                # adapt tensor unpacking to your dataloader output
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(x)
                loss = loss_fn(outputs.logits if hasattr(outputs, "logits") else outputs, y)
                loss.backward()
                optimizer.step()

        return self.get_parameters(), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()

        loss_fn = torch.nn.CrossEntropyLoss()
        loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in self.val_loader:
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)
                outputs = self.model(x)
                logits = outputs.logits if hasattr(outputs, "logits") else outputs
                loss += loss_fn(logits, y).item() * x.size(0)
                preds = logits.argmax(dim=-1)
                correct += (preds == y).sum().item()
                total += x.size(0)

        return float(loss / total), total, {"accuracy": float(correct / total)}

# Server strategy: FedAvg
strategy = fl.server.strategy.FedAvg()

# (Optional) Start Flower server
# fl.server.start_server(
#     server_address="0.0.0.0:8080",
#     config=fl.server.ServerConfig(num_rounds=3),
#     strategy=strategy,
# )

In [ ]:
import flwr as fl

# Launch simulation with 10 virtual clients
fl.simulation.start_simulation(
    client_fn=client_fn, # Function that creates a client instance
    num_clients=10,
    config=fl.server.ServerConfig(num_rounds=5), # Start with 5 rounds
    strategy=fl.server.strategy.FedAvg(),
)